# **LLM Serving with Apigee**


# Redis Semantic Cache

## Install dependencies


In [ ]:
!pip install --upgrade google-genai

## Authenticate your notebook environment (Colab only)
If you are running this notebook on Google Colab, run the following cell to authenticate your environment. This step is not required if you are using Vertex AI Workbench or Colab Enterprise.

In [ ]:
import sys

# Additional authentication is required for Google Colab
if "google.colab" in sys.modules:
    # Authenticate user to Google Cloud
    from google.colab import auth

    auth.authenticate_user()

## Initialize notebook variables

* **PROJECT_ID**: The default GCP project to use when making Vertex API calls.
* **REGION**: The default location to use when making API calls.
* **API_ENDPOINT**:  Desired API endpoint, e.g. https://apigee.iloveapimanagement.com/generate

In [68]:
import time
from google import genai
from google.genai import types
# Define project information
PROJECT_ID = "YOUR_PROJECT_ID"  # @param {type:"string"}
API_ENDPOINT = "https://APIGEE_HOST_NAME/v1/samples/llm-redis-cache"  # @param {type:"string"}
SKIP_CACHE = "false"  # @param {type:"string"}
LOCATION = "global"
MODEL = "gemini-3-flash-preview" # @param {type:"string"}

client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
    http_options=types.HttpOptions(
        api_version='v1',
        base_url=API_ENDPOINT,
        headers = {"x-skipcache": SKIP_CACHE})
)

## Test Sample

Apigee allows you to seamlessly send logs to Cloud Logging using native integration with the [Message Logging](https://cloud.google.com/apigee/docs/api-platform/reference/policies/message-logging-policy#cloudloggingelement) policy. This sample also includes a message chunking solution that allows logging very long messages (ex. 1M tokens supported by Gemini) and connecting them together using a unique message identifier.

With the following cell you'll be able to invoke an LLM and both prompt and candidate resposne will be logged in Cloud Logging.

### Using GenAI SDK

In [ ]:
prompts = ["Why is the sky blue?",
           "What makes the sky blue?",
           "Why does the sky is blue colored?",
           "Can you explain why the sky is blue?",
           "The sky is blue, why is that?"]

for prompt in prompts:
  # 1. get start time
  start_time = time.time()
  response = client.models.generate_content(model=MODEL, contents=prompt)
  # 2. get finish time and the diff
  elapsed_time = time.time() - start_time
  print(response.text)
  print(f"\n\n\n\n### Response Finished with elapsed time {elapsed_time:.2f} seconds ###\n\n\n\n")



### Using GenAI SDK for Stream

In [ ]:
prompts = ["Why is the sky blue?",
           "What makes the sky blue?",
           "Why does the sky is blue colored?",
           "Can you explain why the sky is blue?",
           "The sky is blue, why is that?"]

for prompt in prompts:
  # 1. get start time
  start_time = time.time()
  response = client.models.generate_content_stream(
      model=MODEL, contents=prompt
  )
  for chunk in response:
      print(chunk.text, end="", flush=True)
  # 2. get finish time and the diff
  elapsed_time = time.time() - start_time
  print(f"\n\n\n\n### Stream response Finished with elapsed time {elapsed_time:.2f} seconds ###\n\n\n\n")

